# Reproduce paper Figure 8 — Governance Crossover & Waterfall

One-click reproduction of the paper's Figure 8 from the generated result
CSVs (no model training here — the CSVs come from `run_experiments.py`):

* **(a) The Hierarchical CrossOver Frontier** — margin of the cleaned
  micro-model ceiling (max of LightGBM/MLP) over the RAW deep-model camp,
  vs. the camp MEAN and vs. the camp CHAMPION. 4/4 positive vs mean;
  3/4 vs best (Folsom's LSTM+raw edges out LightGBM+cleaned, shown
  honestly as −0.013).
* **(b) The Causal Governance Waterfall** — Folsom, LightGBM probe:
  label-governance steps are no-ops; the clear-sky anchor features
  deliver the breakthrough.

Data: `results/plots/results_e9_cross_dataset.csv` and
`results/plots/results_e10b_waterfall_met.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = REPO / "results" / "plots"

x = pd.read_csv(RES / "results_e9_cross_dataset.csv")
wf = pd.read_csv(RES / "results_e10b_waterfall_met.csv")
print(f"cross-dataset grid: {len(x)} rows, datasets={sorted(x.dataset.unique())}")
x.head(3)

In [ ]:
# Panel (a) data: cleaned micro-model ceiling vs the RAW deep-model camp.
datasets = ["folsom", "skippd", "sunlake", "gefcom2014"]
disp = ["Folsom", "SKIPP'D", "SunLake", "GEFCom2014"]
vs_mean, vs_best = [], []
for ds in datasets:
    d = x[x.dataset == ds]
    micro = max(d[d.model == m].skill_cleaned.iloc[0]
                for m in ("LightGBM", "MLP"))
    deep = [d[d.model == m].skill_raw.iloc[0]
            for m in ("LSTM", "iTransformer", "PatchTST")]
    vs_mean.append(micro - float(np.mean(deep)))
    vs_best.append(micro - max(deep))
for name, m, b in zip(disp, vs_mean, vs_best):
    print(f"{name:11s} vs mean {m:+.3f}   vs best {b:+.3f}")

In [ ]:
# Panel (b) data: the MET-informed 3-stage waterfall (Folsom, LightGBM).
m = wf[wf.waterfall == "main_3stage"].set_index("stage")["mae"]
mae = [float(m["base"]), float(m["+hygiene"]), float(m["+clear_sky"])]
steps = ["Base (Target-lag + 5 MET)", "+Hygiene", "+Clear-sky"]
for s, v in zip(steps, mae):
    print(f"{s:26s} MAE = {v:.5f}")

In [ ]:
BLUE, ORANGE = "#005ab5", "#dc7d28"

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.6),
                         gridspec_kw={"wspace": 0.28})

# ---- (a) crossover frontier ------------------------------------------
ax = axes[0]
xpos = np.arange(len(disp))
b1 = ax.bar(xpos - 0.19, vs_mean, width=0.35, facecolor=BLUE,
            label=r"vs. deep-model $\mu$")
b2 = ax.bar(xpos + 0.19, vs_best, width=0.35, facecolor=ORANGE,
            edgecolor=ORANGE, linewidth=0.5, hatch="//",
            label=r"vs. deep-model best")
ax.axhline(0.0, color="black", linewidth=0.8)
for bars, vals, col in ((b1, vs_mean, BLUE), (b2, vs_best, ORANGE)):
    for b, v in zip(bars, vals):
        va = "bottom" if v >= 0 else "top"
        off = 0.006 * (1 if v >= 0 else -1)
        ax.text(b.get_x() + b.get_width() / 2, v + off, f"{v:+.3f}",
                ha="center", va=va, fontsize=8.5, color=col,
                fontweight="bold")
all_vals = vs_mean + vs_best
lo, hi = min(all_vals), max(all_vals)
span = hi - lo
ax.set_ylim(lo - 0.22 * span, hi + 0.22 * span)
ax.set_xticks(xpos, disp)
ax.set_ylabel(r"CrossOver Margin $\chi$ (Skill Difference)")
ax.grid(axis="y", linestyle="--", alpha=0.15)
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=10, loc="upper left")
ax.text(0.01, 0.02,
        r"$\chi=\max(\mathrm{LGBM},\mathrm{MLP})_{\mathrm{cleaned}}"
        r"-\mathrm{deep}_{\mathrm{raw}}$",
        transform=ax.transAxes, fontsize=9.5, color="0.25")

# ---- (b) governance waterfall -----------------------------------------
ax = axes[1]
colors = ["#08306b", "#2171b5", ORANGE]
bars = ax.bar(np.arange(3), mae, width=0.55, facecolor=colors,
              edgecolor="none")
ax.axhline(0.0, color="black", linewidth=0.8)
span = max(mae) - min(mae)
for i, (b, v) in enumerate(zip(bars, mae)):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.22 * span, f"{v:.5f}",
            ha="center", va="bottom", fontsize=8.5, color="black")
for i in (1, 2):
    pct = (mae[i] / mae[i - 1] - 1) * 100
    ax.annotate("", xy=(i, mae[i]), xytext=(i - 1, mae[i - 1]),
                arrowprops=dict(arrowstyle="->", color=ORANGE, lw=1.0,
                                connectionstyle="arc3,rad=-0.15"))
    band = 0.0 if i == 2 else 0.14
    ax.text(i - 0.5, max(mae[i], mae[i - 1]) + band * span,
            f"{pct:+.2f}%", ha="center", va="bottom", fontsize=8.5,
            color=ORANGE, fontweight="bold")
ax.set_xticks(np.arange(3), steps, fontsize=9.5)
ax.set_ylabel(r"Daytime Forecast Error (MAE)")
ax.set_ylim(0.04, max(mae) + 0.42 * span)
ax.grid(axis="y", linestyle="--", alpha=0.15)
ax.set_axisbelow(True)

for ax, tag in zip(axes, ("(a)", "(b)")):
    ax.text(-0.22, 1.05, tag, transform=ax.transAxes, fontsize=14,
            fontweight="bold", va="bottom", ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.945))
plt.show()

The rendered figure matches `figures/paper/fig08.png` byte-for-byte in
data (same CSVs, same geometry). To regenerate the shipped asset:
```bash
python scripts/plots/plot_fig08.py
```